---
#☢️ **WORK IN PROGRESS**
This is a notebook consistent of documentation and ideas for our project.

---

#⭐ **NOTES**

A few starting notes before we begin:

No, I won't be bothered to include this **_now_** in the Word document.

We don't have enough time to be sure if this even fits to be deployed, and this documentation alone would require a chapter of its own, or a complete rewrite of the Word document.

If you feel brave, however, be my guest. We lack proper team segmentation so you're practically hopeless. Do note, I will **not** allow any AI summary of this notebook.

As of February 15th, I did NOT include comments or texts showcasing what each cell does.

---

#🤖 **PROJECT**

Our AI project will be multiclass classification. Why? We have an Indicator of Compromise that will be labelled one of three clusters: low, medium, or high.

The datasets are up to debate, but it might be data from Kaggle, or other organizations.

---




#1️⃣ **IMPORTS**

Look, but don't touch. We are importing functionalities we want to use.

In [ ]:
import pandas as pd
import numpy as np
import re
import ipaddress
import hashlib
from urllib.parse import urlparse
from collections import Counter
from pathlib import Path
from datetime import datetime
import json

# We ignore non-critical warnings
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score
)

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    logger.info("SMOTE available")
except ImportError:
    logger.warning("imblearn not available, install with: pip install imbalanced-learn")
    SMOTE_AVAILABLE = False
    ImbPipeline = Pipeline

import joblib

# DO NOT TOUCH
SEED = 42
MODEL_DIR = Path('ioc_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Libraries imported successfully")
logger.info(f"Random seed: {SEED}")
logger.info(f"Model directory: {MODEL_DIR}")
logger.info(f"Timestamp: {datetime.now()}")

In [ ]:
class IoC_FeatureExtractor:
    def __init__(self):
        self.suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.work', '.click']
        self.suspicious_keywords = ['admin', 'login', 'secure', 'account', 'update', 'verify',
                                   'confirm', 'banking', 'paypal', 'microsoft', 'apple']

    def extract_ip_features(self, ip_str):
        import ipaddress
        import numpy as np

        features = {}
        try:
            ip = ipaddress.ip_address(ip_str)
            features['is_ipv4'] = 1 if ip.version == 4 else 0
            features['is_ipv6'] = 1 if ip.version == 6 else 0
            features['is_private'] = 1 if ip.is_private else 0
            features['is_reserved'] = 1 if ip.is_reserved else 0
            features['is_loopback'] = 1 if ip.is_loopback else 0
            features['is_multicast'] = 1 if ip.is_multicast else 0

            if ip.version == 4:
                ip_int = int(ip)
                octets = [
                    (ip_int >> 24) & 0xFF,
                    (ip_int >> 16) & 0xFF,
                    (ip_int >> 8) & 0xFF,
                    ip_int & 0xFF
                ]

                features['first_octet'] = octets[0]
                features['last_octet'] = octets[3]
                features['octet_variance'] = np.var(octets)
                features['sequential_octets'] = sum([
                    1 for i in range(3) if octets[i+1] - octets[i] == 1
                ])

                features['in_suspicious_range'] = 1 if octets[0] in [
                    1, 27, 31, 37, 41, 42, 58, 59, 60, 61, 62
                ] else 0

            # IPv6
            else:
                features.update({
                    k: 0 for k in ['first_octet', 'last_octet', 'octet_variance',
                                   'sequential_octets', 'in_suspicious_range']
                })

        # Invalid IP
        except:
            features = {
                k: 0 for k in ['is_ipv4', 'is_ipv6', 'is_private', 'is_reserved',
                              'is_loopback', 'is_multicast', 'first_octet',
                              'last_octet', 'octet_variance', 'sequential_octets',
                              'in_suspicious_range']
            }
            features['is_invalid'] = 1

        return features

    def extract_domain_features(self, domain):
        import pandas as pd
        import numpy as np

        features = {}
        if not domain or pd.isna(domain):
            return {k: 0 for k in ['domain_length', 'num_dots', 'num_digits', 'num_hyphens',
                                   'has_suspicious_tld', 'has_suspicious_keyword',
                                   'entropy', 'consonant_ratio', 'vowel_ratio', 'digit_ratio']}

        domain = str(domain).lower()
        features['domain_length'] = len(domain)
        features['num_dots'] = domain.count('.')
        features['num_digits'] = sum(c.isdigit() for c in domain)
        features['num_hyphens'] = domain.count('-')
        features['num_underscores'] = domain.count('_')
        features['has_suspicious_tld'] = any(domain.endswith(tld) for tld in self.suspicious_tlds)
        features['has_suspicious_keyword'] = any(kw in domain for kw in self.suspicious_keywords)

        prob = [float(domain.count(c)) / len(domain) for c in dict.fromkeys(list(domain))]
        features['entropy'] = -sum([p * np.log2(p) for p in prob])

        vowels = 'aeiou'
        consonants = 'bcdfghjklmnpqrstvwxyz'
        alpha_chars = [c for c in domain if c.isalpha()]
        if alpha_chars:
            features['vowel_ratio'] = sum(c in vowels for c in alpha_chars) / len(alpha_chars)
            features['consonant_ratio'] = sum(c in consonants for c in alpha_chars) / len(alpha_chars)
        else:
            features['vowel_ratio'] = 0
            features['consonant_ratio'] = 0

        features['digit_ratio'] = features['num_digits'] / features['domain_length'] if features['domain_length'] > 0 else 0
        return features

    def extract_url_features(self, url):
        import pandas as pd
        import ipaddress
        from urllib.parse import urlparse

        features = {}
        if not url or pd.isna(url):
            return {k: 0 for k in ['url_length', 'path_length', 'num_params', 'num_fragments',
                                   'has_ip', 'uses_https', 'num_subdomains', 'has_port',
                                   'num_special_chars', 'has_at_symbol', 'has_double_slash']}

        url = str(url)
        try:
            parsed = urlparse(url)
            features['url_length'] = len(url)
            features['path_length'] = len(parsed.path)
            features['num_params'] = len(parsed.query.split('&')) if parsed.query else 0
            features['num_fragments'] = 1 if parsed.fragment else 0

            try:
                ipaddress.ip_address(parsed.netloc.split(':')[0])
                features['has_ip'] = 1
            except:
                features['has_ip'] = 0

            features['uses_https'] = 1 if parsed.scheme == 'https' else 0
            features['num_subdomains'] = parsed.netloc.count('.') - 1 if '.' in parsed.netloc else 0
            features['has_port'] = 1 if ':' in parsed.netloc else 0

            special_chars = '@%&=+$,;'
            features['num_special_chars'] = sum(url.count(c) for c in special_chars)
            features['has_at_symbol'] = 1 if '@' in url else 0
            features['has_double_slash'] = 1 if '//' in parsed.path else 0
        except:
            features = {k: 0 for k in ['url_length', 'path_length', 'num_params', 'num_fragments',
                                       'has_ip', 'uses_https', 'num_subdomains', 'has_port',
                                       'num_special_chars', 'has_at_symbol', 'has_double_slash']}
        return features

    def extract_hash_features(self, hash_str):
        import pandas as pd
        import numpy as np
        from collections import Counter

        features = {}
        if not hash_str or pd.isna(hash_str):
            return {k: 0 for k in ['hash_length', 'is_md5', 'is_sha1', 'is_sha256',
                                   'char_distribution_entropy', 'hex_digit_ratio']}

        hash_str = str(hash_str).lower()
        features['hash_length'] = len(hash_str)
        features['is_md5'] = 1 if len(hash_str) == 32 else 0
        features['is_sha1'] = 1 if len(hash_str) == 40 else 0
        features['is_sha256'] = 1 if len(hash_str) == 64 else 0

        if hash_str:
            char_counts = Counter(hash_str)
            prob = [count / len(hash_str) for count in char_counts.values()]
            features['char_distribution_entropy'] = -sum([p * np.log2(p) for p in prob if p > 0])
        else:
            features['char_distribution_entropy'] = 0

        hex_chars = set('0123456789abcdef')
        valid_hex = sum(c in hex_chars for c in hash_str)
        features['hex_digit_ratio'] = valid_hex / len(hash_str) if len(hash_str) > 0 else 0
        return features

    def extract_all_features(self, ioc_value, ioc_type):
        from urllib.parse import urlparse

        all_features = {}
        all_features['type_ip'] = 1 if ioc_type == 'ip' else 0
        all_features['type_domain'] = 1 if ioc_type == 'domain' else 0
        all_features['type_url'] = 1 if ioc_type == 'url' else 0
        all_features['type_hash'] = 1 if ioc_type == 'hash' else 0

        if ioc_type == 'ip':
            all_features.update(self.extract_ip_features(ioc_value))
        elif ioc_type == 'domain':
            all_features.update(self.extract_domain_features(ioc_value))
        elif ioc_type == 'url':
            all_features.update(self.extract_url_features(ioc_value))
            try:
                domain = urlparse(str(ioc_value)).netloc
                domain_features = self.extract_domain_features(domain)
                all_features.update({f'url_{k}': v for k, v in domain_features.items()})
            except:
                pass
        elif ioc_type == 'hash':
            all_features.update(self.extract_hash_features(ioc_value))

        return all_features

In [ ]:
# NOTE: I couldn't figure out how to get real data. This is a placeholder.
def create_synthetic_ioc_dataset(n_samples=10000):
    np.random.seed(SEED)
    data = []

    for _ in range(n_samples):
        ioc_type = np.random.choice(['ip', 'domain', 'url', 'hash'], p=[0.3, 0.3, 0.25, 0.15])

        if ioc_type == 'ip':
            risk = np.random.choice(['low', 'medium', 'high'], p=[0.5, 0.3, 0.2])
            if risk == 'high':
                first_octet = np.random.choice([1, 27, 31, 37, 41, 42, 58, 59, 60, 61, 62, 185, 188])
                ip = f"{first_octet}.{np.random.randint(0, 256)}.{np.random.randint(0, 256)}.{np.random.randint(1, 256)}"
            elif risk == 'medium':
                ip = f"{np.random.choice([13, 52, 54, 104, 151, 172])}.{np.random.randint(0, 256)}.{np.random.randint(0, 256)}.{np.random.randint(1, 256)}"
            else:
                ip = f"{np.random.choice([8, 20, 40, 64, 128, 192])}.{np.random.randint(0, 256)}.{np.random.randint(0, 256)}.{np.random.randint(1, 256)}"
            value = ip

        elif ioc_type == 'domain':
            risk = np.random.choice(['low', 'medium', 'high'], p=[0.5, 0.3, 0.2])
            if risk == 'high':
                length = np.random.randint(8, 20)
                domain_name = ''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'), length))
                tld = np.random.choice(['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top'])
                value = domain_name + tld
            elif risk == 'medium':
                base = np.random.choice(['paypal', 'microsoft', 'apple', 'google', 'amazon'])
                suffix = np.random.choice(['secure', 'login', 'verify', 'account', 'update'])
                value = f"{base}-{suffix}.com"
            else:
                prefixes = ['cloud', 'tech', 'soft', 'data', 'web', 'net', 'app']
                suffixes = ['corp', 'inc', 'solutions', 'systems', 'tech', 'labs']
                value = f"{np.random.choice(prefixes)}{np.random.choice(suffixes)}.com"

        elif ioc_type == 'url':
            risk = np.random.choice(['low', 'medium', 'high'], p=[0.5, 0.3, 0.2])
            if risk == 'high':
                protocol = np.random.choice(['http', 'https'], p=[0.7, 0.3])
                ip = f"{np.random.randint(1, 256)}.{np.random.randint(0, 256)}.{np.random.randint(0, 256)}.{np.random.randint(1, 256)}"
                path = ''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'), 15))
                value = f"{protocol}://{ip}/{path}.exe"
            elif risk == 'medium':
                domain = np.random.choice(['bit.ly', 'tinyurl.com', 'goo.gl'])
                short_code = ''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'), 6))
                value = f"https://{domain}/{short_code}"
            else:
                domain = f"{''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz'), 8))}.com"
                value = f"https://{domain}/page"

        else:
            risk = np.random.choice(['low', 'medium', 'high'], p=[0.5, 0.3, 0.2])
            hash_type = np.random.choice(['md5', 'sha1', 'sha256'], p=[0.3, 0.3, 0.4])
            if hash_type == 'md5':
                value = ''.join(np.random.choice(list('0123456789abcdef'), 32))
            elif hash_type == 'sha1':
                value = ''.join(np.random.choice(list('0123456789abcdef'), 40))
            else:
                value = ''.join(np.random.choice(list('0123456789abcdef'), 64))

        data.append({'ioc_value': value, 'ioc_type': ioc_type, 'risk_level': risk})

    return pd.DataFrame(data)

logger.info("Generating IoC dataset...")
df = create_synthetic_ioc_dataset(n_samples=10000)

logger.info(f"Dataset created: {len(df)} samples")
logger.info(f"Risk distribution:\n{df['risk_level'].value_counts()}")
logger.info(f"IoC type distribution:\n{df['ioc_type'].value_counts()}")
print("\nSample data:")
print(df.head(10))


Sample data:
                                           ioc_value ioc_type risk_level
0                              huswkkx9xcvbx3bu6l.ml   domain       high
1                                 cloudsolutions.com   domain        low
2                          https://pooslwty.com/page      url        low
3                                      151.72.166.18       ip     medium
4  89413beb6bc7e2d031731d55935ce19b19d3defe7d6b8d...     hash       high
5                                     59.128.235.136       ip       high
6                                      1.135.162.163       ip       high
7                          https://gzigihlb.com/page      url        low
8                                      192.215.36.99       ip        low
9                                       softcorp.com   domain        low


In [ ]:
logger.info("Extracting features...")
extractor = IoC_FeatureExtractor()

feature_list = []
for idx, row in df.iterrows():
    features = extractor.extract_all_features(row['ioc_value'], row['ioc_type'])
    feature_list.append(features)

features_df = pd.DataFrame(feature_list).fillna(0)

logger.info(f"Features extracted: {features_df.shape[1]} features")

le = LabelEncoder()
y = le.fit_transform(df['risk_level'])

logger.info(f"Label encoding: {dict(enumerate(le.classes_))}")

X = features_df.values
feature_names = list(features_df.columns)

logger.info(f"Feature matrix: {X.shape}")
logger.info(f"Target vector: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

logger.info(f"Training set: {X_train.shape[0]} samples")
logger.info(f"Test set: {X_test.shape[0]} samples")
logger.info(f"Class distribution in train: {np.bincount(y_train)}")
logger.info(f"Class distribution in test: {np.bincount(y_test)}")

In [ ]:
logger.info("Building model pipelines...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def create_pipeline(classifier, use_smote=False):
    if use_smote and SMOTE_AVAILABLE:
        return ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=SEED)),
            ('classifier', classifier)
        ])
    else:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', classifier)
        ])

classifiers = {
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=7,
        min_samples_split=5,
        subsample=0.8,
        random_state=SEED
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=5,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        multi_class='multinomial',
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf',
        C=10,
        class_weight='balanced',
        probability=True,
        random_state=SEED
    )
}

pipelines = {name: create_pipeline(clf, use_smote=True) for name, clf in classifiers.items()}

logger.info(f"Created {len(pipelines)} model pipelines")
logger.info(f"SMOTE enabled: {SMOTE_AVAILABLE}")

In [ ]:
logger.info("Starting hyperparameter tuning for Random Forest...")

param_distributions = {
    'classifier__n_estimators': [200, 300, 400, 500],
    'classifier__max_depth': [20, 30, 40, 50],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None]
}

rf_pipeline = pipelines['Random Forest']

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=SEED,
    verbose=1
)

random_search.fit(X_train, y_train)

logger.info(f"Best parameters: {random_search.best_params_}")
logger.info(f"Best CV score: {random_search.best_score_:.4f}")

best_rf_pipeline = random_search.best_estimator_
pipelines['Random Forest (Tuned)'] = best_rf_pipeline

print("\nBest Random Forest parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best Random Forest parameters:
  classifier__n_estimators: 500
  classifier__min_samples_split: 2
  classifier__min_samples_leaf: 1
  classifier__max_features: None
  classifier__max_depth: 20


In [ ]:
logger.info("Training individual models")

results = {}

for name, pipeline in pipelines.items():
    logger.info(f"Training {name}...")

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_weighted')

    results[name] = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }

    logger.info(f"{name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, CV: {cv_scores.mean():.4f}")

results_df = pd.DataFrame(results).T.sort_values('f1', ascending=False)
print("\n" + "="*80)
print("Model Performance Summary:")
print("="*80)
print(results_df.to_string())


Model Performance Summary:
                       accuracy        f1  precision  recall   cv_mean    cv_std
Gradient Boosting        0.8920  0.892528   0.894442  0.8920  0.906108  0.006493
Random Forest (Tuned)    0.8890  0.889648   0.891967  0.8890  0.904559  0.005410
Extra Trees              0.8875  0.887992   0.889634  0.8875  0.900956  0.007629
Random Forest            0.8835  0.884055   0.886243  0.8835  0.900031  0.006126
SVM                      0.8195  0.821061   0.832526  0.8195  0.813899  0.011145
Logistic Regression      0.7845  0.785777   0.802521  0.7845  0.781820  0.007807


In [ ]:
logger.info("Creating soft voting ensemble...")

top_models = results_df.head(5).index.tolist()
logger.info(f"Ensemble models: {top_models}")

ensemble_estimators = []
for name in top_models:
    if name in pipelines:
        ensemble_estimators.append((name.lower().replace(' ', '_'), pipelines[name]))

voting_clf = VotingClassifier(
    estimators=ensemble_estimators,
    voting='soft',
    n_jobs=-1
)

ensemble_pipeline = voting_clf

logger.info("Training ensemble...")
ensemble_pipeline.fit(X_train, y_train)

logger.info("Ensemble training complete")

In [ ]:
logger.info("Evaluating ensemble...")

y_pred_ensemble = ensemble_pipeline.predict(X_test)
y_pred_proba_ensemble = ensemble_pipeline.predict_proba(X_test)

# Metrics
ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)
ensemble_f1 = f1_score(y_test, y_pred_ensemble, average='weighted')
ensemble_precision = precision_score(y_test, y_pred_ensemble, average='weighted')
ensemble_recall = recall_score(y_test, y_pred_ensemble, average='weighted')

# Per-class metrics
precision_per_class = precision_score(y_test, y_pred_ensemble, average=None)
recall_per_class = recall_score(y_test, y_pred_ensemble, average=None)
f1_per_class = f1_score(y_test, y_pred_ensemble, average=None)

print("\n" + "="*80)
print("ENSEMBLE MODEL PERFORMANCE")
print("="*80)
print(f"Accuracy:  {ensemble_accuracy:.4f}")
print(f"F1 Score:  {ensemble_f1:.4f}")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall:    {ensemble_recall:.4f}")

print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_pred_ensemble, target_names=le.classes_))

print("\n" + "="*80)
print("CONFUSION MATRIX")
print("="*80)
cm = confusion_matrix(y_test, y_pred_ensemble)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

best_individual = results_df.iloc[0]
improvement = ensemble_f1 - best_individual['f1']

print("\n" + "="*80)
print("ENSEMBLE vs BEST INDIVIDUAL")
print("="*80)
print(f"Best Individual: {results_df.index[0]}")
print(f"  F1 Score: {best_individual['f1']:.4f}")
print(f"\nEnsemble:")
print(f"  F1 Score: {ensemble_f1:.4f}")
print(f"\nImprovement: {improvement:+.4f} ({improvement/best_individual['f1']*100:+.2f}%)")

logger.info(f"Ensemble F1 score: {ensemble_f1:.4f}")


ENSEMBLE MODEL PERFORMANCE
Accuracy:  0.8850
F1 Score:  0.8857
Precision: 0.8882
Recall:    0.8850

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        high       0.81      0.90      0.85       409
         low       0.93      0.88      0.91       973
      medium       0.87      0.88      0.88       618

    accuracy                           0.89      2000
   macro avg       0.87      0.89      0.88      2000
weighted avg       0.89      0.89      0.89      2000


CONFUSION MATRIX
        high  low  medium
high     370   22      17
low       53  857      63
medium    35   40     543

ENSEMBLE vs BEST INDIVIDUAL
Best Individual: Gradient Boosting
  F1 Score: 0.8925

Ensemble:
  F1 Score: 0.8857

Improvement: -0.0068 (-0.77%)


In [ ]:
logger.info("Saving model artifacts...")

metadata = {
    # 2.0 because this is very different from the notebook showcased thursday
    'model_version': '2.0',
    'training_date': datetime.now().isoformat(),
    'random_seed': SEED,
    'n_samples': len(df),
    'n_features': len(feature_names),
    'classes': le.classes_.tolist(),
    'ensemble_metrics': {
        'accuracy': float(ensemble_accuracy),
        'f1_score': float(ensemble_f1),
        'precision': float(ensemble_precision),
        'recall': float(ensemble_recall)
    },
    'per_class_metrics': {
        'precision': precision_per_class.tolist(),
        'recall': recall_per_class.tolist(),
        'f1': f1_per_class.tolist()
    },
    'individual_models': {k: {ki: float(vi) for ki, vi in v.items()} for k, v in results.items()},
    'ensemble_models': top_models,
    'smote_used': SMOTE_AVAILABLE,
    'feature_names': feature_names
}

model_artifact = {
    'ensemble_pipeline': ensemble_pipeline,
    'label_encoder': le,
    'feature_extractor': extractor,
    'feature_names': feature_names,
    'metadata': metadata
}

artifact_path = MODEL_DIR / 'ioc_classifier.joblib'
joblib.dump(model_artifact, artifact_path, compress=3)

logger.info(f"Model artifact saved: {artifact_path}")
logger.info(f"Artifact size: {artifact_path.stat().st_size / 1024 / 1024:.2f} MB")

metadata_path = MODEL_DIR / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info(f"Metadata saved: {metadata_path}")

print("\n" + "="*80)
print("MODEL ARTIFACTS SAVED")
print("="*80)
print(f"Main artifact:  {artifact_path}")
print(f"Metadata:       {metadata_path}")
print(f"Model version:  {metadata['model_version']}")
print(f"Training date:  {metadata['training_date']}")


MODEL ARTIFACTS SAVED
Main artifact:  ioc_models/ioc_classifier.joblib
Metadata:       ioc_models/model_metadata.json
Model version:  2.0
Training date:  2026-02-15T10:57:01.626967


In [ ]:
class IoC_Classifier:
    def __init__(self, artifact_path):
        """Load consolidated artifact"""
        logger.info(f"Loading model from {artifact_path}")

        artifact = joblib.load(artifact_path)

        self.pipeline = artifact['ensemble_pipeline']
        self.label_encoder = artifact['label_encoder']
        self.feature_extractor = artifact['feature_extractor']
        self.feature_names = artifact['feature_names']
        self.metadata = artifact['metadata']

        logger.info(f"Model version: {self.metadata['model_version']}")
        logger.info(f"Training date: {self.metadata['training_date']}")
        logger.info(f"F1 score: {self.metadata['ensemble_metrics']['f1_score']:.4f}")

    def predict(self, ioc_value, ioc_type):
        """Predict risk level for single IoC"""
        features = self.feature_extractor.extract_all_features(ioc_value, ioc_type)

        features_df = pd.DataFrame([features])
        features_df = features_df.reindex(columns=self.feature_names, fill_value=0)
        features_df = features_df.fillna(0)

        X = features_df.values
        prediction = self.pipeline.predict(X)[0]
        probabilities = self.pipeline.predict_proba(X)[0]

        risk_level = self.label_encoder.inverse_transform([prediction])[0]

        classes = list(self.label_encoder.classes_)
        prob_dict = {cls: float(prob) for cls, prob in zip(classes, probabilities)}

        return {
            'ioc_value': ioc_value,
            'ioc_type': ioc_type,
            'predicted_risk': risk_level,
            'confidence': float(max(probabilities)),
            'probabilities': prob_dict,
            'model_version': self.metadata['model_version']
        }

    def predict_batch(self, iocs_df):
        feature_list = []
        for idx, row in iocs_df.iterrows():
            features = self.feature_extractor.extract_all_features(
                row['ioc_value'], row['ioc_type']
            )
            feature_list.append(features)

        features_df = pd.DataFrame(feature_list)
        features_df = features_df.reindex(
            columns=self.feature_names, fill_value=0
        )

        # Ensure all NaNs are filled, as reindex might not fill pre-existing NaNs in all cases
        features_df = features_df.fillna(0)

        X = features_df.values

        predictions = self.pipeline.predict(X)
        probabilities = self.pipeline.predict_proba(X)

        results = []
        classes = list(self.label_encoder.classes_)

        for i, (idx, row) in enumerate(iocs_df.iterrows()):
            risk_level = self.label_encoder.inverse_transform([predictions[i]])[0]
            prob_dict = {cls: float(prob) for cls, prob in zip(classes, probabilities[i])}

            results.append({
                'ioc_value': row['ioc_value'],
                'ioc_type': row['ioc_type'],
                'predicted_risk': risk_level,
                'confidence': float(max(probabilities[i])),
                'probabilities': prob_dict
            })

        return pd.DataFrame(results)

clf = IoC_Classifier(artifact_path)
logger.info("classifier initialized")

In [ ]:
logger.info("Testing")

test_iocs = [
    {'ioc_value': '185.220.101.1', 'ioc_type': 'ip'},
    {'ioc_value': '8.8.8.8', 'ioc_type': 'ip'},
    {'ioc_value': 'microsoft-secure.tk', 'ioc_type': 'domain'},
    {'ioc_value': 'google.com', 'ioc_type': 'domain'},
]

print("\n" + "="*80)
print("INFERENCE TESTS")
print("="*80)

for ioc in test_iocs:
    result = clf.predict(ioc['ioc_value'], ioc['ioc_type'])
    print(f"\nIoC: {result['ioc_value']}")
    print(f"Type: {result['ioc_type']}")
    print(f"Risk: {result['predicted_risk'].upper()}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Probabilities: {', '.join([f'{k}={v:.2%}' for k, v in result['probabilities'].items()])}")
    print("-" * 80)

test_df = pd.DataFrame(test_iocs)
batch_results = clf.predict_batch(test_df)

print("\nBatch Prediction Results:")
print(batch_results[['ioc_value', 'predicted_risk', 'confidence']].to_string(index=False))


INFERENCE TESTS

IoC: 185.220.101.1
Type: ip
Risk: HIGH
Confidence: 67.08%
Probabilities: high=67.08%, low=23.65%, medium=9.27%
--------------------------------------------------------------------------------

IoC: 8.8.8.8
Type: ip
Risk: LOW
Confidence: 82.94%
Probabilities: high=0.94%, low=82.94%, medium=16.12%
--------------------------------------------------------------------------------

IoC: microsoft-secure.tk
Type: domain
Risk: HIGH
Confidence: 54.51%
Probabilities: high=54.51%, low=3.80%, medium=41.70%
--------------------------------------------------------------------------------

IoC: google.com
Type: domain
Risk: LOW
Confidence: 91.18%
Probabilities: high=3.42%, low=91.18%, medium=5.40%
--------------------------------------------------------------------------------

Batch Prediction Results:
          ioc_value predicted_risk  confidence
      185.220.101.1           high    0.670779
            8.8.8.8            low    0.829414
microsoft-secure.tk           high    0.5

In [ ]:
print("\nModel Performance:")
print(f"  Accuracy:  {ensemble_accuracy:.4f}")
print(f"  F1 Score:  {ensemble_f1:.4f}")
print(f"  Precision: {ensemble_precision:.4f}")
print(f"  Recall:    {ensemble_recall:.4f}")

print("\nSaved Artifacts:")
print(f"  {artifact_path}")
print(f"  {metadata_path}")

logger.info("Complete")


Model Performance:
  Accuracy:  0.8850
  F1 Score:  0.8857
  Precision: 0.8882
  Recall:    0.8850

Saved Artifacts:
  ioc_models/ioc_classifier.joblib
  ioc_models/model_metadata.json
